# register-back-fn-after-wrap — faded example 2: Register the Backward for Division at argnum=1 (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`. Running the beacon reports progress on the `Backprop: register back fn` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For `div(x, y) = x / y`, the partial derivative with respect to `y` (argnum=1) is `-x / y^2`. When registering this back function, you provide it alongside `add_back_func(t.div, 1, div_back1)`. Both `x` and `y` are available to the back function through its argument list, even though it only differentiates with respect to `y`.

## Faded exercise 2

The `BackwardFuncLookup` and `div_back0` (gradient w.r.t. `x`) are already implemented. Your task is to **implement `div_back1`** — the backward function for `torch.div` at argnum=1 (gradient w.r.t. `y`).

Recall: if `out = x / y`, then `d(out)/d(y) = -x / y^2`.

The signature is `div_back1(grad_out, out, x, y) -> Tensor`.

**Fill in:** Compute the gradient of x/y with respect to y: return -grad_out * x divided by y squared.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def div_back0(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    # d(x/y)/dx = 1/y
    return grad_out / y

def div_back1(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    raise NotImplementedError()  # TODO: Compute the gradient of x/y with respect to y: return -grad_out * x divided by y squared.

def register_div(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.div, 0, div_back0)
    BACK_FUNCS.add_back_func(t.div, 1, div_back1)


def _test():
    import torch as t
    x = t.tensor([6.0, 8.0, 10.0])
    y = t.tensor([2.0, 4.0, 5.0])
    out = x / y
    grad_out = t.ones_like(x)
    result = div_back1(grad_out, out, x, y)
    expected = -x / (y ** 2)
    assert t.allclose(result, expected, atol=1e-6), f'Expected {expected}, got {result}'
    BACK_FUNCS = BackwardFuncLookup()
    register_div(BACK_FUNCS)
    fn = BACK_FUNCS.get_back_func(t.div, 1)
    assert fn is div_back1


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def div_back0(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    return grad_out / y

def div_back1(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    return -grad_out * x / (y ** 2)

def register_div(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.div, 0, div_back0)
    BACK_FUNCS.add_back_func(t.div, 1, div_back1)
```
</details>